# Workshop: Análisis de Sensibilidad de Hiperparámetros
## Sistema Inteligente de Priorización de Cobranzas para PYMES — Grupo 3

**Modelo principal:** XGBoost multiclase  
**Métrica principal:** F1-macro  
**Unidad experimental:** cortes temporales de factura, con separación por `factura_id`  
**Objetivo:** evaluar cómo los hiperparámetros de XGBoost afectan el rendimiento, la generalización y el espacio de búsqueda recomendado.

> Notebook para ejecución reproducible en Google Colab con `Run all`. No contiene métricas inventadas: todos los resultados, tablas, gráficos y conclusiones se generan al ejecutar las celdas.


---
# 1. Contexto metodológico y alineación con la rúbrica

Este notebook responde al workshop de **Análisis de sensibilidad de hiperparámetros**. La rúbrica exige definir claramente el caso de estudio, ejecutar múltiples combinaciones, analizar sensibilidad individual, construir ranking de importancia, estudiar interacciones, tomar decisiones de tuning, generar visualizaciones claras, mantener reproducibilidad y citar referencias en APA 7.

Para evitar fuga de información, el split se realiza por `factura_id`, porque una misma factura puede aparecer en varios cortes temporales. Los identificadores, fechas crudas y variables posteriores al corte se excluyen de los predictores.


---
# 2. Instalación, repositorio y librerías


In [ ]:
# ============================================================
# 2.1 Instalación de dependencias para Google Colab
# ============================================================
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = [
    ('xgboost', 'xgboost'),
    ('sklearn', 'scikit-learn'),
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('matplotlib', 'matplotlib'),
    ('seaborn', 'seaborn'),
    ('scipy', 'scipy'),
    ('joblib', 'joblib')
]

for import_name, pip_name in REQUIRED_PACKAGES:
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f'Instalando {pip_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name, '-q'])

print('✅ Dependencias verificadas.')


In [ ]:
# ============================================================
# 2.2 Clonar/actualizar repositorio oficial del proyecto
# ============================================================
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/kevinUbe23/GRUPO_3.git'
REPO_DIR = Path('GRUPO_3')

if not REPO_DIR.exists():
    result = subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(f'No se pudo clonar el repositorio. Error:\n{result.stderr}')
    print(f'✅ Repositorio clonado en: {REPO_DIR}/')
else:
    result = subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], text=True, capture_output=True)
    print(f'✅ Repositorio existente actualizado en: {REPO_DIR}/')

print('\n📂 Archivos principales detectados en el repositorio:')
for root, dirs, files in os.walk(REPO_DIR):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = Path(root).relative_to(REPO_DIR).parts
    if len(level) > 2:
        continue
    indent = '  ' * len(level)
    print(f'{indent}{Path(root).name}/')
    for f in files[:12]:
        print(f'{indent}  {f}')


In [ ]:
# ============================================================
# 2.3 Importación de librerías y configuración global
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import os, json, time, random, itertools, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr

from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix
)
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except Exception:
    StratifiedGroupKFold = None
    HAS_STRATIFIED_GROUP_KFOLD = False
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, FunctionTransformer
from sklearn.utils.class_weight import compute_sample_weight

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

OUTPUT_DIR = Path('outputs_sensibilidad')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuración de ejecución. Ajusta N_EXPERIMENTS_TARGET si Colab tiene pocos recursos.
N_EXPERIMENTS_TARGET = 48
USE_GROUP_CV = False
N_CV_SPLITS = 3
N_JOBS_XGB = 2

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'font.family': 'DejaVu Sans'
})
sns.set_theme(style='whitegrid')

print('✅ Librerías importadas.')
print(f'   Output dir: {OUTPUT_DIR.resolve()}')
print(f'   RANDOM_STATE: {RANDOM_STATE}')
print(f'   USE_GROUP_CV: {USE_GROUP_CV} | N_CV_SPLITS: {N_CV_SPLITS}')


---
# 3. Carga y validación del dataset

El notebook prioriza `features_ml_prepared.csv` si existe, porque corresponde al dataset preparado. Si no existe, usa `features_ml.csv` del repositorio. No se generan datos nuevos.


In [ ]:
# ============================================================
# 3.1 Utilidades de búsqueda de archivos del proyecto
# ============================================================
SEARCH_ROOTS = [REPO_DIR, Path('/content'), Path('.')]
PRIORITY_FILES = ['features_ml_prepared.csv', 'features_ml.csv']

# Si el usuario subió files.zip en Colab, lo descomprime solo como respaldo local del proyecto.
# No genera datos nuevos; únicamente extrae archivos ya existentes del proyecto.
for possible_zip in [Path('files.zip'), Path('/content/files.zip')]:
    if possible_zip.exists():
        import zipfile
        extract_dir = Path('/content/project_files_from_zip') if Path('/content').exists() else Path('project_files_from_zip')
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(possible_zip, 'r') as zf:
            zf.extractall(extract_dir)
        SEARCH_ROOTS.append(extract_dir)
        print(f'ℹ️  Se detectó y extrajo {possible_zip} en {extract_dir}')

def find_file(filename, roots=SEARCH_ROOTS):
    for base in roots:
        if not Path(base).exists():
            continue
        for root, _, files in os.walk(base):
            if filename in files:
                return Path(root) / filename
    return None

MAIN_DATA_PATH = None
for fname in PRIORITY_FILES:
    candidate = find_file(fname)
    if candidate is not None:
        MAIN_DATA_PATH = candidate
        break

TRAIN_IDS_PATH = find_file('train_facturas_ids.csv')
TEST_IDS_PATH = find_file('test_facturas_ids.csv')
META_PATH = find_file('preprocessing_metadata.json')

if MAIN_DATA_PATH is None:
    raise FileNotFoundError(
        'No se encontró features_ml_prepared.csv ni features_ml.csv en el repositorio o archivos cargados. '
        'Verifica que el repo oficial contenga el dataset o sube el ZIP de datos del proyecto.'
    )

print(f'✅ Dataset principal: {MAIN_DATA_PATH}')
print(f'   train ids       : {TRAIN_IDS_PATH}')
print(f'   test ids        : {TEST_IDS_PATH}')
print(f'   metadata        : {META_PATH}')


In [ ]:
# ============================================================
# 3.2 Cargar dataset y metadatos
# ============================================================
df = pd.read_csv(MAIN_DATA_PATH)

if META_PATH is not None:
    with open(META_PATH, 'r', encoding='utf-8') as f:
        META = json.load(f)
else:
    META = {}

print(f'📊 Shape dataset: {df.shape}')
print(f'📌 Columnas: {list(df.columns)}')
print('\nTipos de datos:')
print(df.dtypes.value_counts())
print('\nNulos detectados:')
nulls = df.isna().sum().sort_values(ascending=False)
print(nulls[nulls > 0].head(20) if (nulls > 0).any() else 'Sin nulos detectados.')


In [ ]:
# ============================================================
# 3.3 Detección robusta del target y de factura_id
# ============================================================
TARGET_CANDIDATES = [
    'target_mora', 'target', 'mora_label', 'estado_mora', 'clase_mora',
    'tramo_mora', 'estado_cartera', 'estado_final_pago'
]

FACTURA_ID_CANDIDATES = ['factura_id', 'id_factura', 'invoice_id']
CLIENTE_ID_CANDIDATES = ['cliente_id', 'id_cliente', 'customer_id']

TARGET_COL = next((c for c in TARGET_CANDIDATES if c in df.columns), None)
FACTURA_ID_COL = next((c for c in FACTURA_ID_CANDIDATES if c in df.columns), None)
CLIENTE_ID_COL = next((c for c in CLIENTE_ID_CANDIDATES if c in df.columns), None)

if TARGET_COL is None:
    raise ValueError(f'No se encontró variable objetivo. Candidatos buscados: {TARGET_CANDIDATES}')
if FACTURA_ID_COL is None:
    raise ValueError(
        'No se encontró factura_id. Para este proyecto no se hará split por fila; '
        'se requiere una columna de factura para evitar fuga entre cortes temporales.'
    )

print(f'✅ Target detectado     : {TARGET_COL}')
print(f'✅ factura_id detectado : {FACTURA_ID_COL}')
print(f'ℹ️  cliente_id detectado : {CLIENTE_ID_COL}')

print('\n📊 Distribución original del target:')
dist_target = pd.DataFrame({
    'conteo': df[TARGET_COL].value_counts(dropna=False),
    'porcentaje': (df[TARGET_COL].value_counts(normalize=True, dropna=False) * 100).round(2)
})
print(dist_target)


---
# 4. Control anti-leakage, codificación del target y split por factura


In [ ]:
# ============================================================
# 4.1 Selección de features sin leakage
# ============================================================
EXACT_EXCLUDE = {
    TARGET_COL.lower(),
    'target', 'label', 'clase_real', 'estado_final', 'dias_mora_final',
    'monto_pagado', 'pago_real', 'resultado_cobro', 'fecha_cierre',
    'cobro_efectivo'
}

# Variables válidas aunque tengan lógica temporal, siempre que estén disponibles al corte.
SAFE_TEMPORAL_FEATURES = {
    'dias_transcurridos_corte', 'esta_vencida_al_corte', 'dias_mora_observable'
}

def leakage_reason(col):
    c = col.lower()
    if c in SAFE_TEMPORAL_FEATURES:
        return None
    if c in EXACT_EXCLUDE:
        return 'exclusión exacta target/leakage'
    if '_id' in c:
        return 'identificador con _id'
    if 'fecha' in c:
        return 'fecha cruda'
    if c in {'target_encoded', 'target_codificado'}:
        return 'target derivado'
    return None

exclude_records = []
for col in df.columns:
    reason = leakage_reason(col)
    if reason is not None:
        exclude_records.append({'columna': col, 'motivo': reason})

excluded_cols = [r['columna'] for r in exclude_records]
FEATURE_COLS = [c for c in df.columns if c not in excluded_cols]

if TARGET_COL in FEATURE_COLS:
    FEATURE_COLS.remove(TARGET_COL)

if not FEATURE_COLS:
    raise ValueError('La selección anti-leakage dejó 0 predictores. Revisa columnas del dataset.')

df_excluded = pd.DataFrame(exclude_records).sort_values(['motivo', 'columna'])
df_excluded.to_csv(OUTPUT_DIR / 'columnas_excluidas_leakage.csv', index=False, encoding='utf-8-sig')

print('🚫 Columnas excluidas del modelado:')
print(df_excluded.to_string(index=False))
print(f'\n✅ Total de features retenidas: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

valid_temporal_present = [c for c in FEATURE_COLS if c.lower() in SAFE_TEMPORAL_FEATURES]
print(f'\n🟢 Variables temporales válidas retenidas: {valid_temporal_present}')


In [ ]:
# ============================================================
# 4.2 Codificación segura del target con LabelEncoder SIEMPRE
# ============================================================
y_raw = df[TARGET_COL].astype(str).fillna('SIN_TARGET')
le_target = LabelEncoder()
y_all = le_target.fit_transform(y_raw)
CLASS_NAMES = list(le_target.classes_)
N_CLASSES = len(CLASS_NAMES)

class_mapping = pd.DataFrame({
    'clase_original': CLASS_NAMES,
    'clase_codificada': list(range(N_CLASSES))
})
class_mapping.to_csv(OUTPUT_DIR / 'mapeo_target_labelencoder.csv', index=False, encoding='utf-8-sig')

expected_classes = np.arange(N_CLASSES)
assert np.array_equal(np.unique(y_all), expected_classes), 'Las clases codificadas no son consecutivas.'

print('✅ Target codificado con LabelEncoder.')
print(class_mapping.to_string(index=False))
print(f'Clases consecutivas en y: {np.unique(y_all).tolist()}')


In [ ]:
# ============================================================
# 4.3 Split por factura_id: usa IDs oficiales o crea split por factura
# ============================================================
def read_id_file(path, id_col):
    ids_df = pd.read_csv(path)
    if id_col in ids_df.columns:
        return ids_df[id_col].dropna().unique()
    # Respaldo: tomar primera columna si el CSV oficial no conservó el nombre.
    return ids_df.iloc[:, 0].dropna().unique()

if TRAIN_IDS_PATH is not None and TEST_IDS_PATH is not None:
    train_ids = read_id_file(TRAIN_IDS_PATH, FACTURA_ID_COL)
    test_ids = read_id_file(TEST_IDS_PATH, FACTURA_ID_COL)
    split_source = 'archivos oficiales train_facturas_ids.csv y test_facturas_ids.csv'
else:
    unique_facturas = df[FACTURA_ID_COL].dropna().unique()
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
    dummy_y = np.zeros(len(df))
    train_idx_tmp, test_idx_tmp = next(splitter.split(df, dummy_y, groups=df[FACTURA_ID_COL]))
    train_ids = df.iloc[train_idx_tmp][FACTURA_ID_COL].dropna().unique()
    test_ids = df.iloc[test_idx_tmp][FACTURA_ID_COL].dropna().unique()
    split_source = 'split nuevo por factura_id con GroupShuffleSplit 80/20'
    pd.DataFrame({FACTURA_ID_COL: train_ids}).to_csv(OUTPUT_DIR / 'train_facturas_ids.csv', index=False)
    pd.DataFrame({FACTURA_ID_COL: test_ids}).to_csv(OUTPUT_DIR / 'test_facturas_ids.csv', index=False)

train_ids = pd.Series(train_ids).dropna().unique()
test_ids = pd.Series(test_ids).dropna().unique()
intersection_ids = set(train_ids).intersection(set(test_ids))
if len(intersection_ids) > 0:
    raise ValueError(f'Leakage detectado: {len(intersection_ids)} factura_id aparecen en train y test.')

train_mask = df[FACTURA_ID_COL].isin(train_ids)
test_mask = df[FACTURA_ID_COL].isin(test_ids)

if train_mask.sum() == 0 or test_mask.sum() == 0:
    raise ValueError('El split por factura_id generó train o test vacío. Revisa los archivos de IDs.')

idx_train = np.where(train_mask.values)[0]
idx_test = np.where(test_mask.values)[0]

X_all_df = df[FEATURE_COLS].copy()
X_train_df = X_all_df.iloc[idx_train].reset_index(drop=True)
X_test_df = X_all_df.iloc[idx_test].reset_index(drop=True)
y_train = y_all[idx_train]
y_test = y_all[idx_test]
groups_train = df.iloc[idx_train][FACTURA_ID_COL].reset_index(drop=True).values
groups_test = df.iloc[idx_test][FACTURA_ID_COL].reset_index(drop=True).values

split_report = {
    'fuente_split': split_source,
    'n_facturas_train': int(len(train_ids)),
    'n_facturas_test': int(len(test_ids)),
    'n_filas_train': int(len(idx_train)),
    'n_filas_test': int(len(idx_test)),
    'interseccion_factura_id': int(len(intersection_ids))
}
with open(OUTPUT_DIR / 'reporte_split_factura_id.json', 'w', encoding='utf-8') as f:
    json.dump(split_report, f, indent=2, ensure_ascii=False)

print('✅ Split por factura_id aplicado correctamente.')
for k, v in split_report.items():
    print(f'{k:25s}: {v}')
print('\nDistribución y_train codificada:', dict(zip(*np.unique(y_train, return_counts=True))))
print('Distribución y_test codificada :', dict(zip(*np.unique(y_test, return_counts=True))))
print('\nMapeo de clases:')
print(class_mapping.to_string(index=False))


In [ ]:
# ============================================================
# 4.4 Split interno de validación por factura dentro de TRAIN
# ============================================================
# Este split interno evita seleccionar hiperparámetros usando el test externo.
# El test externo queda reservado para diagnóstico y evaluación final.
inner_splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
fit_idx, val_idx = next(inner_splitter.split(X_train_df, y_train, groups=groups_train))

X_fit_df = X_train_df.iloc[fit_idx].reset_index(drop=True)
X_val_df = X_train_df.iloc[val_idx].reset_index(drop=True)
y_fit = y_train[fit_idx]
y_val = y_train[val_idx]
groups_fit = groups_train[fit_idx]
groups_val = groups_train[val_idx]

inner_overlap = set(groups_fit).intersection(set(groups_val))
if len(inner_overlap) > 0:
    raise ValueError(f'Leakage interno detectado: {len(inner_overlap)} factura_id en fit y validación.')

inner_report = {
    'n_facturas_fit': int(len(np.unique(groups_fit))),
    'n_facturas_val': int(len(np.unique(groups_val))),
    'n_filas_fit': int(len(X_fit_df)),
    'n_filas_val': int(len(X_val_df)),
    'interseccion_factura_id_fit_val': int(len(inner_overlap))
}
with open(OUTPUT_DIR / 'reporte_split_interno_validacion.json', 'w', encoding='utf-8') as f:
    json.dump(inner_report, f, indent=2, ensure_ascii=False)

print('✅ Split interno por factura_id para tuning aplicado correctamente.')
for k, v in inner_report.items():
    print(f'{k:35s}: {v}')


In [ ]:
# ============================================================
# 4.5 Preprocesamiento seguro: se ajusta SOLO con train/fold
# ============================================================
# Variables categóricas: object/category/bool y codificaciones nominales conocidas.
FORCE_CATEGORICAL_NAMES = {'ultimo_resultado_enc', 'sector_enc'}

cat_cols = []
num_cols = []
for col in FEATURE_COLS:
    dtype = X_train_df[col].dtype
    if str(dtype) in ['object', 'category', 'bool'] or col.lower() in FORCE_CATEGORICAL_NAMES:
        cat_cols.append(col)
    else:
        num_cols.append(col)

# Asegurar que cualquier columna no numérica quede como categórica.
for col in FEATURE_COLS:
    if col not in cat_cols and col not in num_cols:
        cat_cols.append(col)

def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

def make_preprocessor():
    numeric_pipe = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median'))
    ])
    categorical_pipe = Pipeline(steps=[
        # Columnas categóricas numéricas como sector_enc se convierten a object
        # antes de imputar, para permitir una categoría explícita SIN_DATO.
        ('to_object', FunctionTransformer(lambda x: x.astype(object), feature_names_out='one-to-one')),
        ('imputer', SimpleImputer(strategy='constant', fill_value='SIN_DATO')),
        ('onehot', make_onehot_encoder())
    ])
    return ColumnTransformer(
        transformers=[
            ('num', numeric_pipe, num_cols),
            ('cat', categorical_pipe, cat_cols)
        ],
        remainder='drop',
        verbose_feature_names_out=False
    )

print('✅ Preprocesamiento definido sin ajuste global previo al split.')
print(f'   Numéricas    ({len(num_cols)}): {num_cols}')
print(f'   Categóricas  ({len(cat_cols)}): {cat_cols}')
print('ℹ️  El imputer y OneHotEncoder se ajustan dentro de train o dentro de cada fold.')


In [ ]:
# ============================================================
# 4.6 Funciones comunes: modelo XGBoost, métricas y pesos
# ============================================================
def make_xgb_classifier(params=None):
    params = params or {}
    base = dict(
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=RANDOM_STATE,
        tree_method='hist',
        verbosity=0,
        n_jobs=N_JOBS_XGB,
        num_class=N_CLASSES
    )
    base.update(params)
    return XGBClassifier(**base)

def make_pipeline(params=None):
    return Pipeline(steps=[
        ('preprocess', make_preprocessor()),
        ('model', make_xgb_classifier(params))
    ])

def safe_auc(model, X_eval, y_eval):
    try:
        y_prob = model.predict_proba(X_eval)
        if y_prob.shape[1] != N_CLASSES:
            return np.nan
        return roc_auc_score(y_eval, y_prob, labels=list(range(N_CLASSES)), multi_class='ovr', average='macro')
    except Exception:
        return np.nan

def evaluate_model(model, X_eval, y_eval, prefix=''):
    y_pred = model.predict(X_eval)
    metrics = {
        f'{prefix}f1_macro': f1_score(y_eval, y_pred, average='macro', labels=list(range(N_CLASSES)), zero_division=0),
        f'{prefix}accuracy': accuracy_score(y_eval, y_pred),
        f'{prefix}balanced_accuracy': balanced_accuracy_score(y_eval, y_pred),
        f'{prefix}auc_ovr': safe_auc(model, X_eval, y_eval)
    }
    recalls = recall_score(y_eval, y_pred, average=None, labels=list(range(N_CLASSES)), zero_division=0)
    for i, rec in enumerate(recalls):
        metrics[f'{prefix}recall_{CLASS_NAMES[i]}'] = rec
    return metrics, y_pred

print('✅ Funciones de modelado y evaluación definidas.')
print('✅ Los pesos de clase se calcularán después del split por factura_id.')


---
# 5. Modelo base XGBoost con pesos de clase

El modelo base sirve como referencia. Los pesos se calculan **después** del split por factura, usando solo `y_train`, para evitar fuga de información desde test.


In [ ]:
# ============================================================
# 5.1 Entrenamiento del modelo base con sample_weight
# ============================================================
BASE_PARAMS = {
    'n_estimators': 200,
    'max_depth': 4,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'gamma': 0.1,
    'reg_lambda': 2.0
}

sample_weight_train = compute_sample_weight(class_weight='balanced', y=y_train)

print('=' * 70)
print('MODELO BASE XGBOOST — con sample_weight balanceado')
print('=' * 70)
print('Pesos calculados solo con y_train, después del split por factura_id.')
print(pd.Series(sample_weight_train).describe().round(4))

t0 = time.time()
base_model = make_pipeline(BASE_PARAMS)
base_model.fit(X_train_df, y_train, model__sample_weight=sample_weight_train)
base_time = time.time() - t0

base_train_metrics, y_pred_base_train = evaluate_model(base_model, X_train_df, y_train, prefix='train_')
base_test_metrics, y_pred_base_test = evaluate_model(base_model, X_test_df, y_test, prefix='test_')

BASE_F1_TRAIN = base_train_metrics['train_f1_macro']
BASE_F1_TEST = base_test_metrics['test_f1_macro']
BASE_GAP = BASE_F1_TRAIN - BASE_F1_TEST

BASE_METRICS = {
    **base_train_metrics,
    **base_test_metrics,
    'brecha_train_test': BASE_GAP,
    'tiempo_entrenamiento_seg': base_time
}

print('\n📊 Métricas del modelo base:')
for k, v in BASE_METRICS.items():
    print(f'{k:32s}: {v:.4f}' if isinstance(v, (int, float, np.floating)) and not pd.isna(v) else f'{k:32s}: {v}')

print('\n📋 Reporte de clasificación del modelo base en TEST:')
print(classification_report(y_test, y_pred_base_test, labels=list(range(N_CLASSES)), target_names=CLASS_NAMES, zero_division=0))


---
# 6. Diseño y ejecución de experimentos

Se evalúan ocho hiperparámetros: `learning_rate`, `max_depth`, `n_estimators`, `subsample`, `colsample_bytree`, `min_child_weight`, `gamma` y `reg_lambda`. La generación combina variaciones individuales y combinaciones aleatorias para cubrir sensibilidad e interacciones.


In [ ]:
# ============================================================
# 6.1 Espacio de búsqueda y generación reproducible de combinaciones
# ============================================================
PARAM_GRID = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [2, 3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.10, 0.20],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5, 10],
    'gamma': [0, 0.1, 0.5, 1.0, 2.0],
    'reg_lambda': [0.5, 1.0, 2.0, 5.0, 10.0]
}
HP_NUMERIC = list(PARAM_GRID.keys())

BASELINE_FOR_GRID = BASE_PARAMS.copy()

# 1) Variaciones individuales para asegurar cobertura de cada valor.
ofat_configs = []
for hp, values in PARAM_GRID.items():
    for value in values:
        cfg = BASELINE_FOR_GRID.copy()
        cfg[hp] = value
        ofat_configs.append(cfg)

# 2) Combinaciones aleatorias para interacciones.
all_configs = [dict(zip(PARAM_GRID.keys(), vals)) for vals in itertools.product(*PARAM_GRID.values())]
rng = np.random.default_rng(RANDOM_STATE)
random_indices = rng.choice(len(all_configs), size=min(len(all_configs), N_EXPERIMENTS_TARGET * 3), replace=False)
random_configs = [all_configs[i] for i in random_indices]

# 3) Unir sin duplicados hasta N_EXPERIMENTS_TARGET.
def cfg_key(cfg):
    return tuple((k, cfg[k]) for k in HP_NUMERIC)

experiment_configs = []
seen = set()
for cfg in ofat_configs + random_configs:
    key = cfg_key(cfg)
    if key not in seen:
        experiment_configs.append(cfg)
        seen.add(key)
    if len(experiment_configs) >= N_EXPERIMENTS_TARGET:
        break

print(f'📐 Espacio total posible         : {len(all_configs):,} combinaciones')
print(f'✅ Experimentos seleccionados    : {len(experiment_configs)}')
print(f'📌 Cobertura aproximada          : {100 * len(experiment_configs) / len(all_configs):.3f}%')
print('\nPrimeras 5 configuraciones:')
for i, cfg in enumerate(experiment_configs[:5], start=1):
    print(f'Exp {i}: {cfg}')


In [ ]:
# ============================================================
# 6.2 Configuración de validación por grupos
# ============================================================
unique_fit_groups = np.unique(groups_fit)
can_use_cv = USE_GROUP_CV and len(unique_fit_groups) >= N_CV_SPLITS

if can_use_cv:
    if HAS_STRATIFIED_GROUP_KFOLD:
        cv_splitter = StratifiedGroupKFold(n_splits=N_CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        cv_mode = f'StratifiedGroupKFold({N_CV_SPLITS}) sobre el subconjunto fit'
    else:
        cv_splitter = GroupKFold(n_splits=N_CV_SPLITS)
        cv_mode = f'GroupKFold({N_CV_SPLITS}) sobre el subconjunto fit'
    print(f'✅ Validación cruzada activada: {cv_mode}')
    print('   sample_weight se calculará dentro de cada fold, solo con y_train del fold.')
else:
    cv_splitter = None
    cv_mode = 'holdout interno por factura_id dentro de train'
    print('✅ Se usará holdout interno por factura_id para seleccionar hiperparámetros.')
    print('   El test externo NO se usa para seleccionar hiperparámetros.')


In [ ]:
# ============================================================
# 6.3 Ejecución de experimentos con pesos de clase
# ============================================================
results = []
sample_weight_fit = compute_sample_weight(class_weight='balanced', y=y_fit)

for exp_id, params in enumerate(experiment_configs, start=1):
    t0 = time.time()

    cv_f1_scores = []
    if cv_splitter is not None:
        # CV opcional: pesos calculados dentro de cada fold.
        for fold, (tr_idx, val_idx) in enumerate(cv_splitter.split(X_fit_df, y_fit, groups_fit), start=1):
            X_tr_fold = X_fit_df.iloc[tr_idx]
            X_val_fold = X_fit_df.iloc[val_idx]
            y_tr_fold = y_fit[tr_idx]
            y_val_fold = y_fit[val_idx]
            w_fold = compute_sample_weight(class_weight='balanced', y=y_tr_fold)

            fold_model = make_pipeline(params)
            fold_model.fit(X_tr_fold, y_tr_fold, model__sample_weight=w_fold)
            pred_val = fold_model.predict(X_val_fold)
            fold_f1 = f1_score(y_val_fold, pred_val, average='macro', labels=list(range(N_CLASSES)), zero_division=0)
            cv_f1_scores.append(fold_f1)

        f1_val = float(np.mean(cv_f1_scores))
        f1_val_std = float(np.std(cv_f1_scores, ddof=1)) if len(cv_f1_scores) > 1 else 0.0

        # Modelo diagnóstico entrenado en el subconjunto fit para medir train/val/test sin usar test en selección.
        exp_model = make_pipeline(params)
        exp_model.fit(X_fit_df, y_fit, model__sample_weight=sample_weight_fit)
    else:
        # Holdout interno: un solo entrenamiento por configuración.
        exp_model = make_pipeline(params)
        exp_model.fit(X_fit_df, y_fit, model__sample_weight=sample_weight_fit)
        pred_val = exp_model.predict(X_val_df)
        f1_val = f1_score(y_val, pred_val, average='macro', labels=list(range(N_CLASSES)), zero_division=0)
        f1_val_std = np.nan

    # Métricas diagnósticas. Test externo se reporta, pero no se usa para elegir BEST_ROW.
    train_metrics, _ = evaluate_model(exp_model, X_fit_df, y_fit, prefix='train_')
    val_metrics, _ = evaluate_model(exp_model, X_val_df, y_val, prefix='val_')
    test_metrics, pred_test = evaluate_model(exp_model, X_test_df, y_test, prefix='test_')

    elapsed = time.time() - t0
    f1_train = train_metrics['train_f1_macro']
    f1_test = test_metrics['test_f1_macro']
    gap_train_val = f1_train - f1_val
    gap_train_test = f1_train - f1_test

    if gap_train_val > 0.15:
        obs = 'Posible sobreajuste: brecha train-validación alta'
    elif f1_val < max(0.30, BASE_F1_TEST * 0.85):
        obs = 'Posible subajuste: F1-macro bajo frente a referencia'
    elif f1_val > BASE_F1_TEST:
        obs = 'Supera al modelo base en validación interna'
    else:
        obs = 'Rendimiento similar o inferior al modelo base'

    row = {
        'experiment_id': exp_id,
        **params,
        'f1_macro_train': round(f1_train, 4),
        'f1_macro_val': round(f1_val, 4),
        'f1_macro_val_std': round(f1_val_std, 4) if not pd.isna(f1_val_std) else np.nan,
        'f1_macro_test_diagnostico': round(f1_test, 4),
        'accuracy_val': round(val_metrics['val_accuracy'], 4),
        'balanced_accuracy_val': round(val_metrics['val_balanced_accuracy'], 4),
        'accuracy_test_diagnostico': round(test_metrics['test_accuracy'], 4),
        'balanced_accuracy_test_diagnostico': round(test_metrics['test_balanced_accuracy'], 4),
        'auc_ovr_test_diagnostico': round(test_metrics['test_auc_ovr'], 4) if not pd.isna(test_metrics['test_auc_ovr']) else np.nan,
        'brecha_train_val': round(gap_train_val, 4),
        'brecha_train_test': round(gap_train_test, 4),
        'tiempo_entrenamiento_seg': round(elapsed, 2),
        'cv_mode': cv_mode,
        'observacion': obs
    }

    for class_name in CLASS_NAMES:
        key = f'test_recall_{class_name}'
        row[key] = round(test_metrics.get(key, np.nan), 4)

    results.append(row)

    if exp_id % 10 == 0 or exp_id == len(experiment_configs):
        print(f'✔ {exp_id}/{len(experiment_configs)} experimentos completados')

print(f'\n✅ Experimentos finalizados: {len(results)}')


---
# 7. Tabla general de resultados y ranking de configuraciones


In [ ]:
# ============================================================
# 7.1 Exportación de resultados generales
# ============================================================
df_results = pd.DataFrame(results).sort_values('f1_macro_val', ascending=False).reset_index(drop=True)
BEST_ROW = df_results.iloc[0].copy()

results_path = OUTPUT_DIR / 'experimentos_sensibilidad_hiperparametros.csv'
df_results.to_csv(results_path, index=False, encoding='utf-8-sig')

print(f'✅ Tabla general exportada: {results_path}')
print(f'Shape: {df_results.shape}')
print('\n📊 Estadísticas F1-macro validación:')
print(df_results['f1_macro_val'].describe().round(4))

print('\n🏆 Mejor configuración por F1-macro validación:')
show_cols = ['experiment_id'] + HP_NUMERIC + ['f1_macro_train','f1_macro_val','f1_macro_test_diagnostico','brecha_train_val','brecha_train_test','observacion']
print(BEST_ROW[show_cols].to_string())

mejora_abs_val = float(BEST_ROW['f1_macro_val'] - BASE_F1_TEST)
mejora_rel_val = float((mejora_abs_val / BASE_F1_TEST) * 100) if BASE_F1_TEST > 0 else np.nan
print(f'\nModelo base F1-macro test: {BASE_F1_TEST:.4f}')
print(f'Mejor F1-macro validación: {BEST_ROW["f1_macro_val"]:.4f}')
print(f'Mejora absoluta vs base   : {mejora_abs_val:+.4f}')
print(f'Mejora relativa vs base   : {mejora_rel_val:+.2f}%')


In [ ]:
# ============================================================
# 7.2 Top 10, bottom 10 y gráfico de ranking
# ============================================================
rank_cols = ['experiment_id'] + HP_NUMERIC + ['f1_macro_val','f1_macro_test_diagnostico','brecha_train_val','observacion']
top10 = df_results[rank_cols].head(10)
bottom10 = df_results[rank_cols].tail(10)

top10_path = OUTPUT_DIR / 'top10_configuraciones.csv'
bottom10_path = OUTPUT_DIR / 'bottom10_configuraciones.csv'
top10.to_csv(top10_path, index=False, encoding='utf-8-sig')
bottom10.to_csv(bottom10_path, index=False, encoding='utf-8-sig')

print('🏅 TOP 10 configuraciones:')
print(top10.to_string(index=False))
print('\n⚠️ BOTTOM 10 configuraciones:')
print(bottom10.to_string(index=False))
print(f'\n✅ Exportados: {top10_path} | {bottom10_path}')

fig, ax = plt.subplots(figsize=(12, 5))
vals = df_results['f1_macro_val'].values
ax.bar(range(len(vals)), vals, alpha=0.85)
ax.axhline(BASE_F1_TEST, linestyle='--', linewidth=2, label=f'Modelo base test ({BASE_F1_TEST:.4f})')
ax.set_xlabel('Experimento ordenado por F1-macro validación')
ax.set_ylabel('F1-macro validación')
ax.set_title('Ranking F1-macro de experimentos de sensibilidad')
ax.legend()
plt.tight_layout()
fig_path = OUTPUT_DIR / 'ranking_f1_macro_experimentos.png'
fig.savefig(fig_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'✅ Guardado: {fig_path}')


---
# 8. Análisis de sensibilidad individual

Cada hiperparámetro se analiza con tabla agrupada, media, mediana, desviación estándar, máximo, mínimo, gráfico tipo PDP/sensibilidad e interpretación automática basada en los resultados reales.


In [ ]:
# ============================================================
# 8.0 Función de sensibilidad individual con interpretación dinámica
# ============================================================
sensitivity_records = []

SENSITIVITY_PLOT_FILES = {
    'learning_rate': 'sensibilidad_learning_rate.png',
    'max_depth': 'sensibilidad_max_depth.png',
    'n_estimators': 'sensibilidad_n_estimators.png',
    'subsample': 'sensibilidad_subsample.png',
    'colsample_bytree': 'sensibilidad_colsample_bytree.png',
    'min_child_weight': 'sensibilidad_min_child_weight.png',
    'gamma': 'sensibilidad_gamma.png',
    'reg_lambda': 'sensibilidad_reg_lambda.png'
}

def analyze_sensitivity(df_res, hp_name, metric='f1_macro_val'):
    grouped = (
        df_res.groupby(hp_name)
        .agg(
            media=(metric, 'mean'),
            mediana=(metric, 'median'),
            std=(metric, 'std'),
            maximo=(metric, 'max'),
            minimo=(metric, 'min'),
            n_exp=(metric, 'count'),
            brecha_media=('brecha_train_val', 'mean')
        )
        .reset_index()
        .sort_values(hp_name)
    )
    grouped[['media','mediana','std','maximo','minimo','brecha_media']] = grouped[['media','mediana','std','maximo','minimo','brecha_media']].round(4)

    best_idx = grouped['media'].idxmax()
    best_value = grouped.loc[best_idx, hp_name]
    best_mean = grouped.loc[best_idx, 'media']
    recommended_values = grouped.loc[grouped['media'] >= best_mean * 0.98, hp_name].tolist()
    worst_value = grouped.loc[grouped['media'].idxmin(), hp_name]

    sorted_means = grouped['media'].values
    saturation_msg = 'No evaluable con un solo valor.'
    if len(sorted_means) >= 3:
        last_delta = abs(sorted_means[-1] - sorted_means[-2])
        total_delta = grouped['media'].max() - grouped['media'].min()
        if total_delta > 0 and last_delta <= total_delta * 0.10:
            saturation_msg = 'Hay señal de saturación: los últimos cambios aportan mejora marginal.'
        else:
            saturation_msg = 'No se observa saturación fuerte en el extremo superior del rango probado.'

    overfit_values = grouped.loc[grouped['brecha_media'] > 0.15, hp_name].tolist()
    underfit_values = grouped.loc[grouped['media'] < max(0.30, BASE_F1_TEST * 0.85), hp_name].tolist()

    recommendation = (
        f'Rango/valores recomendados: {recommended_values}. '
        f'Mejor media observada en {hp_name}={best_value}. '
        f'Evitar valores con bajo desempeño relativo como {worst_value}. '
        f'{saturation_msg} '
        f'Sobreajuste potencial en: {overfit_values if overfit_values else "no destacado"}. '
        f'Subajuste potencial en: {underfit_values if underfit_values else "no destacado"}.'
    )

    grouped['hiperparametro'] = hp_name
    grouped['valor_recomendado_por_media'] = grouped[hp_name].isin(recommended_values)
    grouped['interpretacion'] = recommendation

    print(f'\n📊 Sensibilidad individual: {hp_name}')
    print(grouped.to_string(index=False))
    print(f'\n📝 Interpretación dinámica: {recommendation}')

    # Gráfico tipo PDP/sensibilidad.
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.errorbar(grouped[hp_name].astype(str), grouped['media'], yerr=grouped['std'].fillna(0), marker='o', capsize=4)
    ax.axhline(BASE_F1_TEST, linestyle='--', linewidth=1.5, label=f'Base test ({BASE_F1_TEST:.3f})')
    ax.set_title(f'Sensibilidad de F1-macro frente a {hp_name}')
    ax.set_xlabel(hp_name)
    ax.set_ylabel(f'{metric} promedio')
    ax.legend()
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    fig_path = OUTPUT_DIR / SENSITIVITY_PLOT_FILES[hp_name]
    fig.savefig(fig_path, dpi=160, bbox_inches='tight')
    plt.show()
    print(f'✅ Guardado: {fig_path}')

    return grouped


In [ ]:
# ============================================================
# 8.1 Ejecutar sensibilidad individual para los 8 hiperparámetros
# ============================================================
sensitivity_tables = []
for hp in ['learning_rate', 'max_depth', 'n_estimators', 'subsample',
           'colsample_bytree', 'min_child_weight', 'gamma', 'reg_lambda']:
    sensitivity_tables.append(analyze_sensitivity(df_results, hp))

df_sensitivity = pd.concat(sensitivity_tables, ignore_index=True)
sens_path = OUTPUT_DIR / 'sensibilidad_individual_hiperparametros.csv'
df_sensitivity.to_csv(sens_path, index=False, encoding='utf-8-sig')
print(f'\n✅ Tabla consolidada exportada: {sens_path}')


---
# 9. Ranking de importancia de hiperparámetros

Se combinan tres señales: correlación de Spearman absoluta, variación promedio de F1-macro por hiperparámetro y un modelo surrogate `RandomForestRegressor` que predice F1-macro a partir de hiperparámetros.


In [ ]:
# ============================================================
# 9.1 Spearman + variación promedio + surrogate model
# ============================================================
# 1) Spearman absoluto
spearman_rows = []
for hp in HP_NUMERIC:
    try:
        corr, pval = spearmanr(df_results[hp], df_results['f1_macro_val'])
        if pd.isna(corr):
            corr, pval = 0.0, 1.0
    except Exception:
        corr, pval = 0.0, 1.0
    spearman_rows.append({'hiperparametro': hp, 'spearman_r': corr, 'p_value': pval, 'importancia_estadistica': abs(corr)})
df_spearman = pd.DataFrame(spearman_rows)

# 2) Delta promedio de F1 por hiperparámetro
variation_rows = []
for hp in HP_NUMERIC:
    means = df_results.groupby(hp)['f1_macro_val'].mean()
    variation_rows.append({'hiperparametro': hp, 'delta_f1_promedio': float(means.max() - means.min())})
df_variation = pd.DataFrame(variation_rows)

# 3) Modelo surrogate
X_surr = df_results[HP_NUMERIC].copy()
y_surr = df_results['f1_macro_val'].values
surrogate = RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1, min_samples_leaf=2)
surrogate.fit(X_surr, y_surr)
surrogate_importance = pd.DataFrame({
    'hiperparametro': HP_NUMERIC,
    'importancia_surrogate': surrogate.feature_importances_
})
surrogate_r2 = surrogate.score(X_surr, y_surr)

print(f'✅ Surrogate RandomForestRegressor ajustado. R² in-sample: {surrogate_r2:.4f}')


In [ ]:
# ============================================================
# 9.2 Ranking final combinado e interpretación
# ============================================================
df_rank = (
    df_spearman
    .merge(df_variation, on='hiperparametro', how='left')
    .merge(surrogate_importance, on='hiperparametro', how='left')
)

def normalize(series):
    max_val = series.max()
    if max_val == 0 or pd.isna(max_val):
        return pd.Series(np.zeros(len(series)), index=series.index)
    return series / max_val

df_rank['norm_spearman'] = normalize(df_rank['importancia_estadistica'])
df_rank['norm_delta'] = normalize(df_rank['delta_f1_promedio'])
df_rank['norm_surrogate'] = normalize(df_rank['importancia_surrogate'])

df_rank['score_combinado'] = (
    0.35 * df_rank['norm_spearman'] +
    0.25 * df_rank['norm_delta'] +
    0.40 * df_rank['norm_surrogate']
).round(4)

df_rank = df_rank.sort_values('score_combinado', ascending=False).reset_index(drop=True)
df_rank['ranking_final'] = df_rank.index + 1

def impact_decision(score):
    if score >= 0.65:
        return 'crítico'
    if score >= 0.35:
        return 'moderado'
    return 'bajo impacto'

def tuning_decision(impact):
    if impact == 'crítico':
        return 'mantener amplio y explorar más fino'
    if impact == 'moderado':
        return 'reducir rango alrededor de mejores valores'
    return 'fijar valor o reducir fuertemente'

df_rank['decision_impacto'] = df_rank['score_combinado'].apply(impact_decision)
df_rank['decision_tuning'] = df_rank['decision_impacto'].apply(tuning_decision)
df_rank['interpretacion'] = df_rank.apply(
    lambda r: f"{r['hiperparametro']} tiene impacto {r['decision_impacto']} según score combinado={r['score_combinado']:.4f}.",
    axis=1
)

rank_path = OUTPUT_DIR / 'ranking_importancia_hiperparametros.csv'
df_rank.to_csv(rank_path, index=False, encoding='utf-8-sig')

print('🏆 Ranking final de importancia de hiperparámetros:')
print(df_rank[['ranking_final','hiperparametro','importancia_estadistica','delta_f1_promedio','importancia_surrogate','score_combinado','decision_impacto','decision_tuning']].to_string(index=False))
print(f'\n✅ Exportado: {rank_path}')

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = df_rank.sort_values('score_combinado', ascending=True)
ax.barh(plot_df['hiperparametro'], plot_df['score_combinado'], alpha=0.85)
ax.set_xlabel('Score combinado de importancia')
ax.set_title('Ranking de importancia de hiperparámetros')
ax.axvline(0.65, linestyle='--', linewidth=1.2, label='Crítico')
ax.axvline(0.35, linestyle='--', linewidth=1.2, label='Moderado')
ax.legend()
plt.tight_layout()
fig_path = OUTPUT_DIR / 'ranking_importancia_hiperparametros.png'
fig.savefig(fig_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'✅ Guardado: {fig_path}')


---
# 10. Análisis de interacciones

Se generan heatmaps para cinco pares clave y una interpretación dinámica de mejor combinación, peor combinación, riesgo de sobreajuste y reducción del espacio de búsqueda.


In [ ]:
# ============================================================
# 10.0 Función de heatmap e interpretación dinámica de interacción
# ============================================================
interaction_records = []

INTERACTION_PLOTS = {
    ('learning_rate', 'n_estimators'): 'heatmap_learning_rate_n_estimators.png',
    ('max_depth', 'min_child_weight'): 'heatmap_max_depth_min_child_weight.png',
    ('subsample', 'colsample_bytree'): 'heatmap_subsample_colsample_bytree.png',
    ('max_depth', 'reg_lambda'): 'heatmap_max_depth_reg_lambda.png',
    ('gamma', 'max_depth'): 'heatmap_gamma_max_depth.png'
}

def analyze_interaction(df_res, hp_x, hp_y, metric='f1_macro_val'):
    pivot = df_res.pivot_table(index=hp_y, columns=hp_x, values=metric, aggfunc='mean')
    pivot_gap = df_res.pivot_table(index=hp_y, columns=hp_x, values='brecha_train_val', aggfunc='mean')

    stacked = pivot.stack(dropna=True)
    best_key = stacked.idxmax()
    worst_key = stacked.idxmin()
    best_score = stacked.max()
    worst_score = stacked.min()

    risky = pivot_gap.stack(dropna=True)
    overfit_combos = risky[risky > 0.15].sort_values(ascending=False).head(3)

    best_combo_text = f'{hp_y}={best_key[0]}, {hp_x}={best_key[1]}'
    worst_combo_text = f'{hp_y}={worst_key[0]}, {hp_x}={worst_key[1]}'
    avoid_text = '; '.join([f'{hp_y}={idx[0]}, {hp_x}={idx[1]} (gap={val:.3f})' for idx, val in overfit_combos.items()])
    if not avoid_text:
        avoid_text = worst_combo_text

    interpretation = (
        f'Mejor combinación observada: {best_combo_text} con {metric}={best_score:.4f}. '
        f'Combinación más débil: {worst_combo_text} con {metric}={worst_score:.4f}. '
        f'Combinaciones a evitar por bajo rendimiento o brecha alta: {avoid_text}. '
        f'Para reducir búsqueda, priorizar vecindad de {best_combo_text} y descartar extremos similares a {worst_combo_text}.'
    )

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.heatmap(pivot.round(3), annot=True, fmt='.3f', linewidths=0.5, ax=ax)
    ax.set_title(f'Interacción {hp_x} × {hp_y} — {metric} promedio')
    ax.set_xlabel(hp_x)
    ax.set_ylabel(hp_y)
    plt.tight_layout()
    fig_path = OUTPUT_DIR / INTERACTION_PLOTS[(hp_x, hp_y)]
    fig.savefig(fig_path, dpi=160, bbox_inches='tight')
    plt.show()

    print(f'\n📌 Interacción {hp_x} × {hp_y}')
    print(pivot.round(4).to_string())
    print(f'\n📝 {interpretation}')
    print(f'✅ Guardado: {fig_path}')

    interaction_records.append({
        'interaccion': f'{hp_x} x {hp_y}',
        'hp_x': hp_x,
        'hp_y': hp_y,
        'mejor_combinacion': best_combo_text,
        'mejor_f1_macro': round(float(best_score), 4),
        'peor_combinacion': worst_combo_text,
        'peor_f1_macro': round(float(worst_score), 4),
        'combinaciones_a_evitar': avoid_text,
        'interpretacion': interpretation,
        'archivo_grafico': str(fig_path)
    })


In [ ]:
# ============================================================
# 10.1 Ejecutar las cinco interacciones obligatorias
# ============================================================
for pair in [
    ('learning_rate', 'n_estimators'),
    ('max_depth', 'min_child_weight'),
    ('subsample', 'colsample_bytree'),
    ('max_depth', 'reg_lambda'),
    ('gamma', 'max_depth')
]:
    analyze_interaction(df_results, pair[0], pair[1])

df_interactions = pd.DataFrame(interaction_records)
interactions_path = OUTPUT_DIR / 'interacciones_hiperparametros.csv'
df_interactions.to_csv(interactions_path, index=False, encoding='utf-8-sig')
print(f'\n✅ Interacciones exportadas: {interactions_path}')


---
# 11. Diagnóstico de overfitting / underfitting

Se evalúa F1-macro en entrenamiento, validación y test. La brecha train-validación permite identificar sobreajuste; un F1 bajo frente a la referencia permite identificar subajuste.


In [ ]:
# ============================================================
# 11.1 Diagnóstico train vs validation/test
# ============================================================
OVERFIT_THRESHOLD = 0.15
UNDERFIT_REFERENCE = max(0.30, BASE_F1_TEST * 0.85)

def diagnose(row):
    if row['brecha_train_val'] > OVERFIT_THRESHOLD and row['f1_macro_val'] < UNDERFIT_REFERENCE:
        return 'sobreajuste + subajuste'
    if row['brecha_train_val'] > OVERFIT_THRESHOLD:
        return 'sobreajuste'
    if row['f1_macro_val'] < UNDERFIT_REFERENCE:
        return 'subajuste'
    return 'ajuste razonable'

df_results['diagnostico_generalizacion'] = df_results.apply(diagnose, axis=1)
df_results.to_csv(results_path, index=False, encoding='utf-8-sig')

print('📊 Distribución de diagnósticos:')
print(df_results['diagnostico_generalizacion'].value_counts().to_string())

fig, ax = plt.subplots(figsize=(8, 7))
for diag, grp in df_results.groupby('diagnostico_generalizacion'):
    ax.scatter(grp['f1_macro_train'], grp['f1_macro_val'], label=diag, alpha=0.8, s=60)
lims = [
    min(df_results['f1_macro_train'].min(), df_results['f1_macro_val'].min()) - 0.02,
    max(df_results['f1_macro_train'].max(), df_results['f1_macro_val'].max()) + 0.02
]
ax.plot(lims, lims, linestyle='--', linewidth=1.5, label='train = validación')
ax.scatter(BEST_ROW['f1_macro_train'], BEST_ROW['f1_macro_val'], marker='*', s=220, label='mejor experimento')
ax.scatter(BASE_F1_TRAIN, BASE_F1_TEST, marker='D', s=130, label='modelo base train/test')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('F1-macro entrenamiento')
ax.set_ylabel('F1-macro validación')
ax.set_title('Diagnóstico de overfitting / underfitting')
ax.legend(fontsize=8)
plt.tight_layout()
fig_path = OUTPUT_DIR / 'train_vs_test_overfitting.png'
fig.savefig(fig_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'✅ Guardado: {fig_path}')


In [ ]:
# ============================================================
# 11.2 Tablas de configuraciones con riesgo
# ============================================================
risk_cols = ['experiment_id'] + HP_NUMERIC + ['f1_macro_train','f1_macro_val','f1_macro_test_diagnostico','brecha_train_val','diagnostico_generalizacion']

df_overfits = df_results[df_results['diagnostico_generalizacion'].str.contains('sobreajuste', regex=False)]
df_underfits = df_results[df_results['diagnostico_generalizacion'].str.contains('subajuste', regex=False)]

print(f'⚠️ Configuraciones con sobreajuste: {len(df_overfits)}')
if len(df_overfits) > 0:
    print(df_overfits[risk_cols].head(10).to_string(index=False))

print(f'\n🔵 Configuraciones con subajuste: {len(df_underfits)}')
if len(df_underfits) > 0:
    print(df_underfits[risk_cols].head(10).to_string(index=False))

print('\n🛠️ Recomendación técnica:')
print('Reducir profundidad, aumentar min_child_weight/gamma/reg_lambda o usar subsample/colsample moderados si domina sobreajuste; ampliar capacidad o reducir regularización si domina subajuste.')


---
# 12. Modelo final, matriz de confusión y reporte por clase

El modelo final se entrena con la mejor configuración seleccionada por F1-macro de validación. El test externo se usa para evaluación final, no para seleccionar hiperparámetros.


In [ ]:
# ============================================================
# 12.1 Entrenar mejor modelo y evaluar en test externo
# ============================================================
BEST_PARAMS = {hp: BEST_ROW[hp].item() if hasattr(BEST_ROW[hp], 'item') else BEST_ROW[hp] for hp in HP_NUMERIC}
# Convertir tipos para XGBoost
for k in ['n_estimators','max_depth','min_child_weight']:
    BEST_PARAMS[k] = int(BEST_PARAMS[k])
for k in ['learning_rate','subsample','colsample_bytree','gamma','reg_lambda']:
    BEST_PARAMS[k] = float(BEST_PARAMS[k])

print('🏆 BEST_PARAMS seleccionados por validación:')
print(json.dumps(BEST_PARAMS, indent=2, ensure_ascii=False))

best_model = make_pipeline(BEST_PARAMS)
best_model.fit(X_train_df, y_train, model__sample_weight=sample_weight_train)

best_train_metrics, y_pred_best_train = evaluate_model(best_model, X_train_df, y_train, prefix='train_')
best_test_metrics, y_pred_best_test = evaluate_model(best_model, X_test_df, y_test, prefix='test_')

BEST_FINAL_METRICS = {**best_train_metrics, **best_test_metrics}
BEST_FINAL_METRICS['brecha_train_test'] = BEST_FINAL_METRICS['train_f1_macro'] - BEST_FINAL_METRICS['test_f1_macro']

print('\n📊 Métricas finales del mejor modelo:')
for k, v in BEST_FINAL_METRICS.items():
    print(f'{k:32s}: {v:.4f}' if isinstance(v, (int, float, np.floating)) and not pd.isna(v) else f'{k:32s}: {v}')

mejora_abs_test = BEST_FINAL_METRICS['test_f1_macro'] - BASE_F1_TEST
mejora_rel_test = (mejora_abs_test / BASE_F1_TEST) * 100 if BASE_F1_TEST > 0 else np.nan
print(f'\n📈 Comparación final vs modelo base:')
print(f'F1-macro base test : {BASE_F1_TEST:.4f}')
print(f'F1-macro mejor test: {BEST_FINAL_METRICS["test_f1_macro"]:.4f}')
print(f'Mejora absoluta    : {mejora_abs_test:+.4f}')
print(f'Mejora relativa    : {mejora_rel_test:+.2f}%')


In [ ]:
# ============================================================
# 12.2 Matriz de confusión y reporte por clase
# ============================================================
cm = confusion_matrix(y_test, y_pred_best_test, labels=list(range(N_CLASSES)))
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_df, annot=True, fmt='d', linewidths=0.5, ax=ax)
ax.set_title('Matriz de confusión — mejor modelo XGBoost')
ax.set_xlabel('Predicción')
ax.set_ylabel('Clase real')
plt.tight_layout()
cm_path = OUTPUT_DIR / 'matriz_confusion_mejor_modelo.png'
fig.savefig(cm_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'✅ Matriz guardada: {cm_path}')

report_dict = classification_report(
    y_test, y_pred_best_test,
    labels=list(range(N_CLASSES)),
    target_names=CLASS_NAMES,
    zero_division=0,
    output_dict=True
)
report_df = pd.DataFrame(report_dict).T.reset_index().rename(columns={'index': 'clase'})
report_csv_path = OUTPUT_DIR / 'reporte_metricas_por_clase.csv'
report_df.to_csv(report_csv_path, index=False, encoding='utf-8-sig')
print(f'✅ Reporte CSV guardado: {report_csv_path}')
print(report_df.to_string(index=False))

# Gráfico del reporte por clase: precision, recall, f1-score para clases reales.
class_report_plot = report_df[report_df['clase'].isin(CLASS_NAMES)].copy()
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(class_report_plot))
width = 0.25
ax.bar(x - width, class_report_plot['precision'], width, label='precision')
ax.bar(x, class_report_plot['recall'], width, label='recall')
ax.bar(x + width, class_report_plot['f1-score'], width, label='f1-score')
ax.set_xticks(x)
ax.set_xticklabels(class_report_plot['clase'], rotation=30, ha='right')
ax.set_ylim(0, 1.05)
ax.set_title('Reporte de métricas por clase — mejor modelo')
ax.set_ylabel('Métrica')
ax.legend()
plt.tight_layout()
report_plot_path = OUTPUT_DIR / 'reporte_metricas_por_clase.png'
fig.savefig(report_plot_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'✅ Gráfico guardado: {report_plot_path}')


---
# 13. Reducción del espacio de búsqueda y recomendaciones de tuning


In [ ]:
# ============================================================
# 13.1 Tabla dinámica de recomendaciones de tuning
# ============================================================
recommendation_rows = []

for hp in HP_NUMERIC:
    grp = df_sensitivity[df_sensitivity['hiperparametro'] == hp].copy()
    # La columna del valor conserva el nombre del hiperparámetro.
    hp_values = grp[hp].dropna().tolist()
    recommended_values = grp.loc[grp['valor_recomendado_por_media'] == True, hp].dropna().tolist()
    if not recommended_values:
        recommended_values = [grp.loc[grp['media'].idxmax(), hp]]

    rank_row = df_rank[df_rank['hiperparametro'] == hp].iloc[0]
    impact = rank_row['decision_impacto']
    if impact == 'crítico':
        decision = 'mantener amplio'
    elif impact == 'moderado':
        decision = 'reducir rango'
    else:
        decision = 'fijar valor'

    justification = (
        f"Impacto {impact}; mejor media en {hp}={grp.loc[grp['media'].idxmax(), hp]}; "
        f"valores recomendados por umbral 98% del máximo: {recommended_values}."
    )

    recommendation_rows.append({
        'hiperparametro': hp,
        'rango_probado': str(hp_values),
        'rango_recomendado': str(recommended_values),
        'justificacion': justification,
        'decision': decision
    })

df_tuning_recommendations = pd.DataFrame(recommendation_rows)
tuning_path = OUTPUT_DIR / 'recomendaciones_tuning_hiperparametros.csv'
df_tuning_recommendations.to_csv(tuning_path, index=False, encoding='utf-8-sig')

print('📉 Recomendaciones de reducción del espacio de búsqueda:')
print(df_tuning_recommendations.to_string(index=False))
print(f'\n✅ Exportado: {tuning_path}')


---
# 14. Conclusiones dinámicas basadas en resultados reales

La siguiente celda genera texto final a partir de `df_results`, `df_rank`, `BEST_PARAMS` y las métricas ejecutadas. No contiene conclusiones fijas prefabricadas.


In [ ]:
# ============================================================
# 14.1 Generar conclusiones automáticas desde resultados reales
# ============================================================
top3_hps = df_rank.sort_values('ranking_final').head(3)['hiperparametro'].tolist()
low_impact_hps = df_rank[df_rank['decision_impacto'] == 'bajo impacto']['hiperparametro'].tolist()
overfit_count = int((df_results['diagnostico_generalizacion'].str.contains('sobreajuste', regex=False)).sum())
underfit_count = int((df_results['diagnostico_generalizacion'].str.contains('subajuste', regex=False)).sum())

conclusion_text = f"""
CONCLUSIONES TÉCNICAS DEL ANÁLISIS DE SENSIBILIDAD

1. El modelo base XGBoost obtuvo un F1-macro en test de {BASE_F1_TEST:.4f}. La mejor configuración seleccionada por validación obtuvo F1-macro de validación de {BEST_ROW['f1_macro_val']:.4f} y F1-macro final en test de {BEST_FINAL_METRICS['test_f1_macro']:.4f}.

2. La mejora absoluta final frente al modelo base fue de {mejora_abs_test:+.4f} puntos de F1-macro, equivalente a una mejora relativa de {mejora_rel_test:+.2f}%.

3. Según el ranking combinado real, los tres hiperparámetros con mayor impacto fueron: {', '.join(top3_hps)}. Esta conclusión proviene del score combinado Spearman + variación promedio + modelo surrogate, no de una afirmación prefabricada.

4. Se detectaron {overfit_count} configuraciones con señales de sobreajuste y {underfit_count} configuraciones con señales de subajuste. Las configuraciones de riesgo deben revisarse contra la tabla de overfitting para evitar combinaciones con alta brecha train-validación.

5. Los hiperparámetros de bajo impacto observados fueron: {', '.join(low_impact_hps) if low_impact_hps else 'ninguno bajo el umbral definido'}. Estos pueden fijarse o reducirse si se busca disminuir costo computacional.

6. El espacio de búsqueda recomendado queda documentado en recomendaciones_tuning_hiperparametros.csv. Las decisiones de mantener amplio, reducir rango o fijar valor se basan en el impacto real y la sensibilidad individual ejecutada.

7. Todas las métricas, tablas y gráficos fueron generados por ejecución del notebook, respetando split por factura_id, exclusión de leakage, codificación segura del target y sample_weight calculado después del split.
""".strip()

print(conclusion_text)

conclusion_path = OUTPUT_DIR / 'conclusiones_dinamicas_sensibilidad.txt'
with open(conclusion_path, 'w', encoding='utf-8') as f:
    f.write(conclusion_text)
print(f'\n✅ Conclusiones exportadas: {conclusion_path}')


In [ ]:
# ============================================================
# 14.2 Resumen JSON reproducible
# ============================================================
summary = {
    'proyecto': 'Sistema Inteligente de Priorización de Cobranzas para PYMES',
    'workshop': 'Análisis de sensibilidad de hiperparámetros',
    'modelo_principal': 'XGBoostClassifier',
    'metrica_principal': 'F1-macro',
    'random_state': RANDOM_STATE,
    'dataset_usado': str(MAIN_DATA_PATH),
    'target_col': TARGET_COL,
    'factura_id_col': FACTURA_ID_COL,
    'n_clases': N_CLASSES,
    'mapeo_clases': class_mapping.to_dict(orient='records'),
    'split': split_report,
    'n_experimentos': int(len(df_results)),
    'cv_mode': cv_mode,
    'modelo_base': {
        'params': BASE_PARAMS,
        'f1_macro_train': round(float(BASE_F1_TRAIN), 4),
        'f1_macro_test': round(float(BASE_F1_TEST), 4),
        'brecha_train_test': round(float(BASE_GAP), 4)
    },
    'mejor_modelo': {
        'best_params': BEST_PARAMS,
        'f1_macro_val': round(float(BEST_ROW['f1_macro_val']), 4),
        'f1_macro_test': round(float(BEST_FINAL_METRICS['test_f1_macro']), 4),
        'mejora_abs_test_vs_base': round(float(mejora_abs_test), 4),
        'mejora_rel_test_vs_base_pct': round(float(mejora_rel_test), 2)
    },
    'top3_hiperparametros': top3_hps,
    'ranking_hiperparametros': df_rank.to_dict(orient='records'),
    'archivos_exportados_en': str(OUTPUT_DIR)
}

summary_path = OUTPUT_DIR / 'resumen_sensibilidad_hiperparametros.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=str)
print(f'✅ Resumen JSON exportado: {summary_path}')
print(json.dumps(summary, indent=2, ensure_ascii=False, default=str)[:4000])


---
# 15. Referencias APA 7

Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785–794. https://doi.org/10.1145/2939672.2939785

Bergstra, J., & Bengio, Y. (2012). Random search for hyper-parameter optimization. *Journal of Machine Learning Research, 13*(10), 281–305.

James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An introduction to statistical learning: With applications in Python*. Springer.

He, H., & Garcia, E. A. (2009). Learning from imbalanced data. *IEEE Transactions on Knowledge and Data Engineering, 21*(9), 1263–1284. https://doi.org/10.1109/TKDE.2008.239

Kaufman, S., Rosset, S., Perlich, C., & Stitelman, O. (2012). Leakage in data mining: Formulation, detection, and avoidance. *ACM Transactions on Knowledge Discovery from Data, 6*(4), 1–21. https://doi.org/10.1145/2382577.2382579

Molnar, C. (2022). *Interpretable machine learning: A guide for making black box models explainable* (2nd ed.). Leanpub.


---
# 16. Checklist final contra rúbrica


In [ ]:
# ============================================================
# 16.1 Checklist final de cumplimiento
# ============================================================
expected_graphs = [
    'ranking_f1_macro_experimentos.png',
    'sensibilidad_learning_rate.png',
    'sensibilidad_max_depth.png',
    'sensibilidad_n_estimators.png',
    'sensibilidad_subsample.png',
    'sensibilidad_colsample_bytree.png',
    'sensibilidad_min_child_weight.png',
    'sensibilidad_gamma.png',
    'sensibilidad_reg_lambda.png',
    'ranking_importancia_hiperparametros.png',
    'heatmap_learning_rate_n_estimators.png',
    'heatmap_max_depth_min_child_weight.png',
    'heatmap_subsample_colsample_bytree.png',
    'heatmap_max_depth_reg_lambda.png',
    'heatmap_gamma_max_depth.png',
    'train_vs_test_overfitting.png',
    'matriz_confusion_mejor_modelo.png',
    'reporte_metricas_por_clase.png'
]
expected_tables = [
    'experimentos_sensibilidad_hiperparametros.csv',
    'sensibilidad_individual_hiperparametros.csv',
    'ranking_importancia_hiperparametros.csv',
    'interacciones_hiperparametros.csv',
    'top10_configuraciones.csv',
    'bottom10_configuraciones.csv',
    'resumen_sensibilidad_hiperparametros.json'
]

checklist = [
    ('definición del caso de estudio', True, 'Problema, modelo XGBoost, F1-macro y 8 hiperparámetros documentados.'),
    ('mínimo 3 hiperparámetros', len(HP_NUMERIC) >= 3, f'{len(HP_NUMERIC)} hiperparámetros analizados.'),
    ('múltiples combinaciones', len(df_results) > 10, f'{len(df_results)} combinaciones ejecutadas.'),
    ('tabla estructurada de experimentos', (OUTPUT_DIR / 'experimentos_sensibilidad_hiperparametros.csv').exists(), 'CSV de experimentos generado.'),
    ('sensibilidad individual', (OUTPUT_DIR / 'sensibilidad_individual_hiperparametros.csv').exists(), '8 análisis individuales con gráficos.'),
    ('ranking de importancia', (OUTPUT_DIR / 'ranking_importancia_hiperparametros.csv').exists(), 'Spearman + variación + surrogate.'),
    ('análisis de interacciones', (OUTPUT_DIR / 'interacciones_hiperparametros.csv').exists(), '5 heatmaps obligatorios.'),
    ('interpretación y toma de decisiones', (OUTPUT_DIR / 'recomendaciones_tuning_hiperparametros.csv').exists(), 'Tabla dinámica de reducción del espacio.'),
    ('visualizaciones claras', all((OUTPUT_DIR / f).exists() for f in expected_graphs), 'Gráficos exportados automáticamente.'),
    ('dataset reproducible', len(intersection_ids) == 0, 'Split por factura_id sin intersección.'),
    ('referencias APA', True, 'Sección 15 con referencias APA 7.'),
    ('exclusión de leakage', (OUTPUT_DIR / 'columnas_excluidas_leakage.csv').exists(), 'Identificadores, fechas y variables futuras excluidas.'),
    ('sample_weight post-split', True, 'compute_sample_weight aplicado después del split y dentro de fold.'),
    ('conclusiones dinámicas', (OUTPUT_DIR / 'conclusiones_dinamicas_sensibilidad.txt').exists(), 'Texto generado desde resultados reales.')
]

print('=' * 80)
print('CHECKLIST FINAL — RÚBRICA WORKSHOP SENSIBILIDAD')
print('=' * 80)
all_ok = True
for item, status, evidence in checklist:
    icon = '✅' if status else '❌'
    print(f'{icon} {item}: {evidence}')
    all_ok = all_ok and bool(status)

print('\n📁 Archivos esperados:')
for f in expected_tables + expected_graphs:
    print(f' - {OUTPUT_DIR / f}')

print('\nRESULTADO FINAL:')
print('✅ Notebook listo para entrega.' if all_ok else '⚠️ Revisar criterios marcados con ❌ antes de entregar.')
